In [ ]:
!pip install torchbnn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchbnn as bnn
import numpy as np

In [ ]:
def gaussian_nll_loss(pred_mean, pred_log_var, target):
    """
    Heteroscedastic Gaussian NLL for regression.

    pred_mean:    (B,)
    pred_log_var: (B,)
    target:       (B,)
    """

    target = target.float().view(-1)
    pred_mean = pred_mean.view(-1)
    pred_log_var = pred_log_var.view(-1)

    pred_log_var = torch.clamp(pred_log_var, min=-10.0, max=10.0)

    precision = torch.exp(-pred_log_var)

    loss = 0.5 * (
        precision * (target - pred_mean) ** 2
        + pred_log_var
    )

    return loss.mean()


def calculate_entropy_dist(mu, sigma):
    # Standard normal distribution
    standard_normal = torch.distributions.Normal(loc=0.0, scale=1.0)

    # Compute z = (0 - mu) / sigma
    z = -mu / (sigma + 1e-8)  # tiny epsilon ONLY for division safety

    # CDF of z under standard normal
    p_negative = standard_normal.cdf(z)
    p_positive = 1 - p_negative

    # Create entropy tensor of same shape
    entropy = torch.zeros_like(mu)

    # Mask where p_positive is between (1e-10, 1-1e-10)
    nondeterministic_mask = (p_positive > 1e-10) & (p_positive < 1 - 1e-10)

    # Compute entropy only where it matters
    p_pos = p_positive[nondeterministic_mask]
    p_neg = p_negative[nondeterministic_mask]

    entropy[nondeterministic_mask] = -(p_pos * torch.log2(p_pos) + p_neg * torch.log2(p_neg))

    return entropy

In [ ]:
def build_vgg16_dex_backbone(pretrained=True, freeze_features=False):
    """
    VGG16 / DEX-style feature extractor for age estimation.

    Uses:
        VGG16 features
        VGG16 avgpool
        fc6 + ReLU
        fc7 + ReLU

    Removes default VGG dropout layers to keep uncertainty mechanisms controlled.
    """

    if pretrained:
        weights = models.VGG16_Weights.IMAGENET1K_V1
    else:
        weights = None

    vgg = models.vgg16(weights=weights)

    features = vgg.features
    avgpool = vgg.avgpool

    # Original VGG classifier:
    # 0 Linear(25088 -> 4096)
    # 1 ReLU
    # 2 Dropout
    # 3 Linear(4096 -> 4096)
    # 4 ReLU
    # 5 Dropout
    # 6 Linear(4096 -> 1000)
    #
    # We remove dropout and final classifier.
    classifier_body = nn.Sequential(
        vgg.classifier[0],
        vgg.classifier[1],
        vgg.classifier[3],
        vgg.classifier[4],
    )

    if freeze_features:
        for param in features.parameters():
            param.requires_grad = False

    return features, avgpool, classifier_body, 4096

In [ ]:
class BayesianVGG16_AgeRegression(nn.Module):
    def __init__(
        self,
        drop_type=None,
        drop_prob=0.2,
        alpha=50,
        gamma=0.1,
        pretrained=True,
        freeze_features=False,
        prior_mu=0.0,
        prior_sigma=0.1
    ):
        super().__init__()

        self.features, self.avgpool, self.classifier_body, in_features = build_vgg16_dex_backbone(
            pretrained=pretrained,
            freeze_features=freeze_features
        )

        # Bayesian final regression head:
        # output[0] = mean age
        # output[1] = log variance
        #self.regression_head = bnn.BayesLinear(
        #    prior_mu=prior_mu,
        #    prior_sigma=prior_sigma,
        #    in_features=in_features,
        #    out_features=2
        #)

        self.mean_head = bnn.BayesLinear(
            prior_mu=prior_mu,
            prior_sigma=prior_sigma,
            in_features=in_features,
            out_features=1
        )

        self.logvar_head = nn.Linear(
            in_features,
            1
        )

        self.drop_type = drop_type
        self.drop_prob = drop_prob
        self.alpha = alpha
        self.gamma = gamma

        if self.drop_type == "dropout":
            # Apply dropout on the feature vector, not directly on mean/log_var.
            self.dropout_layer = nn.Dropout(p=self.drop_prob)

    def forward_features(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier_body(x)
        return x

    def forward_old(self, x, current_epoch=None, total_epochs=None):
        x = self.forward_features(x)
        head = self.regression_head

        if self.training:
            if self.drop_type == "dropconnect":
                masked_mu, _ = self.random_dropconnect(
                    head.weight_mu,
                    head.weight_log_sigma
                )

                out = F.linear(
                    x,
                    masked_mu,
                    head.bias_mu if head.bias else None
                )

            elif self.drop_type == "signentropy":
                if current_epoch is None or total_epochs is None:
                    raise ValueError(
                        "current_epoch and total_epochs are required for signentropy training."
                    )

                with torch.no_grad():
                    new_mu, new_log_sigma = self.signentropy_dropconnect_train(
                        head.weight_mu,
                        head.weight_log_sigma,
                        current_epoch,
                        total_epochs
                    )

                    head.weight_mu.copy_(new_mu)
                    head.weight_log_sigma.copy_(new_log_sigma)

                out = head(x)

            else:
                if self.drop_type == "dropout":
                    x = self.dropout_layer(x)

                out = head(x)

        else:
            if self.drop_type == "dropconnect":
                masked_mu = head.weight_mu * (1.0 - self.drop_prob)

                out = F.linear(
                    x,
                    masked_mu,
                    head.bias_mu if head.bias else None
                )

            elif self.drop_type == "signentropy":
                # Use learned attenuated posterior mean for deterministic prediction.
                # For MC posterior sampling, call the MC function separately.
                out = F.linear(
                    x,
                    head.weight_mu,
                    head.bias_mu if head.bias else None
                )

            else:
                out = head(x)

        pred_mean = out[:, 0]
        pred_log_var = out[:, 1]

        return pred_mean, pred_log_var
    def forward(self, x, current_epoch=None, total_epochs=None):
        """
        Split-head regression forward pass.

        mean_head:
            Bayesian head for age mean prediction.
            Sign Entropy / DropConnect are applied only here.

        logvar_head:
            Deterministic head for aleatoric uncertainty.
            Sign Entropy is NOT applied here.
        """

        x_features = self.forward_features(x)

        mean_head = self.mean_head

        # -----------------------------------------------------
        # Mean prediction: Bayesian / Dropout / DropConnect / SignEntropy
        # -----------------------------------------------------
        if self.training:

            if self.drop_type == "dropconnect":
                masked_mu, _ = self.random_dropconnect(
                    mean_head.weight_mu,
                    mean_head.weight_log_sigma
                )

                bias = mean_head.bias_mu if hasattr(mean_head, "bias_mu") else None

                pred_mean = F.linear(
                    x_features,
                    masked_mu,
                    bias
                ).squeeze(1)

            elif self.drop_type == "signentropy":
                if current_epoch is None or total_epochs is None:
                    raise ValueError(
                        "current_epoch and total_epochs are required for signentropy training."
                    )

                with torch.no_grad():
                    new_mu, new_log_sigma = self.signentropy_dropconnect_train(
                        mean_head.weight_mu,
                        mean_head.weight_log_sigma,
                        current_epoch,
                        total_epochs
                    )

                    mean_head.weight_mu.copy_(new_mu)
                    mean_head.weight_log_sigma.copy_(new_log_sigma)

                pred_mean = mean_head(x_features).squeeze(1)

            else:
                x_mean = x_features

                if self.drop_type == "dropout":
                    x_mean = self.dropout_layer(x_mean)

                pred_mean = mean_head(x_mean).squeeze(1)

        else:

            if self.drop_type == "dropconnect":
                masked_mu = mean_head.weight_mu * (1.0 - self.drop_prob)

                bias = mean_head.bias_mu if hasattr(mean_head, "bias_mu") else None

                pred_mean = F.linear(
                    x_features,
                    masked_mu,
                    bias
                ).squeeze(1)

            elif self.drop_type == "signentropy":
                bias = mean_head.bias_mu if hasattr(mean_head, "bias_mu") else None

                pred_mean = F.linear(
                    x_features,
                    mean_head.weight_mu,
                    bias
                ).squeeze(1)

            else:
                # Deterministic posterior-mean prediction for plain/dropout at eval time.
                bias = mean_head.bias_mu if hasattr(mean_head, "bias_mu") else None

                pred_mean = F.linear(
                    x_features,
                    mean_head.weight_mu,
                    bias
                ).squeeze(1)

        # -----------------------------------------------------
        # Aleatoric uncertainty prediction
        # Deterministic log-variance head.
        # Sign Entropy / DropConnect do NOT touch this.
        # -----------------------------------------------------
        pred_log_var = self.logvar_head(x_features).squeeze(1)

        return pred_mean, pred_log_var

    # -----------------------------------------------------
    # Random DropConnect
    # -----------------------------------------------------
    def random_dropconnect(self, mu, log_sigma):
        mask = torch.bernoulli(
            torch.full_like(mu, 1.0 - self.drop_prob)
        )

        mu = mu * mask
        log_sigma = log_sigma * mask + (1.0 - mask) * (-10.0)

        return mu, log_sigma

    # -----------------------------------------------------
    # Sign-Entropy DropConnect
    # -----------------------------------------------------
    def signentropy_dropconnect_train(self, mu, log_sigma, current_epoch, total_epochs):
        sigma = torch.exp(log_sigma)

        entropy = calculate_entropy_dist(mu, sigma)
        E = entropy.flatten()

        E_threshold = torch.quantile(E, 1.0 - self.drop_prob)

        score = self.alpha * (E - E_threshold)
        p_drop = torch.sigmoid(score)
        p_drop = torch.clamp(p_drop, 0.0, 1.0)

        cos_term = 0.5 * (
            1.0 + torch.cos(
                torch.tensor(
                    np.pi * current_epoch / total_epochs,
                    device=mu.device
                )
            )
        )

        annealing_factor = 1.0 - (cos_term ** self.gamma)

        p_drop_annealed = annealing_factor * p_drop
        p_drop_annealed = p_drop_annealed.view_as(mu)

        masked_mu = mu * (1.0 - p_drop_annealed)

        masked_sigma = sigma * (1.0 - p_drop_annealed)
        masked_log_sigma = torch.log(masked_sigma + 1e-8)

        return masked_mu, masked_log_sigma

In [ ]:
class ConcreteVGG16_AgeRegression(nn.Module):
    def __init__(
        self,
        drop_prob=0.2,
        alpha=50,
        pretrained=True,
        freeze_features=False,
        weight_regularizer=1e-6,
        dropout_regularizer=1e-5,
        concrete_temperature=0.1
    ):
        super().__init__()

        self.features, self.avgpool, self.classifier_body, in_features = build_vgg16_dex_backbone(
            pretrained=pretrained,
            freeze_features=freeze_features
        )

        # Deterministic final regression head
        self.regression_head = nn.Linear(in_features, 2)

        self.drop_type = "concrete"
        self.drop_prob = drop_prob
        self.alpha = alpha

        init_p = np.clip(drop_prob, 1e-6, 1.0 - 1e-6)
        init_logit = np.log(init_p) - np.log(1.0 - init_p)

        self.concrete_p_logit = nn.Parameter(
            torch.tensor([init_logit], dtype=torch.float32)
        )

        self.concrete_temperature = concrete_temperature
        self.weight_regularizer = weight_regularizer
        self.dropout_regularizer = dropout_regularizer

        self.cd_regularization = torch.tensor(0.0)

    def forward_features(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier_body(x)
        return x

    def forward(self, x, current_epoch=None, total_epochs=None):
        x = self.forward_features(x)

        input_dimensionality = x[0].numel()

        # Concrete Dropout stays stochastic in both train and eval mode.
        x = self.concrete_dropout(x)

        out = self.regression_head(x)

        self.cd_regularization = self.concrete_dropout_regularization(
            self.regression_head,
            input_dimensionality
        )

        pred_mean = out[:, 0]
        pred_log_var = out[:, 1]

        return pred_mean, pred_log_var

    def concrete_dropout(self, x):
        eps = 1e-7

        p = torch.sigmoid(self.concrete_p_logit)
        p = torch.clamp(p, eps, 1.0 - eps)

        unif_noise = torch.rand_like(x)

        drop_prob = (
            torch.log(p + eps)
            - torch.log(1.0 - p + eps)
            + torch.log(unif_noise + eps)
            - torch.log(1.0 - unif_noise + eps)
        )

        drop_prob = torch.sigmoid(drop_prob / self.concrete_temperature)

        random_tensor = 1.0 - drop_prob
        retain_prob = 1.0 - p

        x = x * random_tensor
        x = x / retain_prob

        return x

    def concrete_dropout_regularization(self, layer, input_dimensionality):
        eps = 1e-7

        p = torch.sigmoid(self.concrete_p_logit)
        p = torch.clamp(p, eps, 1.0 - eps)

        sum_of_square = 0.0
        for param in layer.parameters():
            sum_of_square = sum_of_square + torch.sum(param ** 2)

        weights_regularizer = (
            self.weight_regularizer * sum_of_square / (1.0 - p)
        )

        dropout_regularizer = p * torch.log(p)
        dropout_regularizer = dropout_regularizer + (
            1.0 - p
        ) * torch.log(1.0 - p)

        dropout_regularizer = (
            dropout_regularizer
            * self.dropout_regularizer
            * input_dimensionality
        )

        regularization = weights_regularizer + dropout_regularizer

        return regularization.squeeze()

    def get_concrete_dropout_prob(self):
        return torch.sigmoid(
            self.concrete_p_logit
        ).detach().cpu().item()

In [ ]:
#### Dataset
# ============================================================
# Download UTKFace from Kaggle using kagglehub
# Dataset: jangedoo/utkface-new
# ============================================================

!pip -q install kagglehub

import kagglehub
from pathlib import Path
import os

DATASET_SLUG = "jangedoo/utkface-new"

dataset_path = kagglehub.dataset_download(DATASET_SLUG)
dataset_path = Path(dataset_path)

print("Dataset downloaded to:", dataset_path)
print("Files/folders:")
for p in dataset_path.iterdir():
    print(p)

Using Colab cache for faster access to the 'utkface-new' dataset.
Dataset downloaded to: /kaggle/input/utkface-new
Files/folders:
/kaggle/input/utkface-new/UTKFace
/kaggle/input/utkface-new/utkface_aligned_cropped
/kaggle/input/utkface-new/crop_part1


In [ ]:
# ============================================================
# Collect and parse UTKFace images
# ============================================================

import re
import pandas as pd
import numpy as np
from pathlib import Path

def collect_utkface_dataframe(root_dir):
    root_dir = Path(root_dir)

    image_extensions = ["*.jpg", "*.jpeg", "*.png"]
    image_paths = []

    for ext in image_extensions:
        image_paths.extend(list(root_dir.rglob(ext)))

    print("Total image files found:", len(image_paths))

    rows = []

    for img_path in image_paths:
        filename = img_path.name

        # UTKFace format: age_gender_race_datetime.jpg
        parts = filename.split("_")

        if len(parts) < 4:
            continue

        try:
            age = int(parts[0])
            gender = int(parts[1])
            race = int(parts[2])
        except:
            continue

        # Remove corrupted / strange labels
        if age < 0 or age > 116:
            continue

        rows.append({
            "image_path": str(img_path),
            "file_name": filename,
            "age": age,
            "gender": gender,
            "race": race
        })

    df = pd.DataFrame(rows)

    print("Valid parsed images:", len(df))
    print(df.head())

    return df


utkface_root = dataset_path / "UTKFace"

df = collect_utkface_dataframe(utkface_root)

print("\nAge summary:")
print(df["age"].describe())

print("\nGender counts:")
print(df["gender"].value_counts())

print("\nRace counts:")
print(df["race"].value_counts())

Total image files found: 23708
Valid parsed images: 23705
                                          image_path  \
0  /kaggle/input/utkface-new/UTKFace/26_0_2_20170...   
1  /kaggle/input/utkface-new/UTKFace/22_1_1_20170...   
2  /kaggle/input/utkface-new/UTKFace/21_1_3_20170...   
3  /kaggle/input/utkface-new/UTKFace/28_0_0_20170...   
4  /kaggle/input/utkface-new/UTKFace/17_1_4_20170...   

                               file_name  age  gender  race  
0  26_0_2_20170104023102422.jpg.chip.jpg   26       0     2  
1  22_1_1_20170112233644761.jpg.chip.jpg   22       1     1  
2  21_1_3_20170105003215901.jpg.chip.jpg   21       1     3  
3  28_0_0_20170117180555824.jpg.chip.jpg   28       0     0  
4  17_1_4_20170103222931966.jpg.chip.jpg   17       1     4  

Age summary:
count    23705.000000
mean        33.300907
std         19.885708
min          1.000000
25%         23.000000
50%         29.000000
75%         45.000000
max        116.000000
Name: age, dtype: float64

Gender counts:
g

In [ ]:
# ============================================================
# Train / Val / Test split binning is done to make sure similar sample distriution is there for train, test and val
# ============================================================
"""
from sklearn.model_selection import train_test_split

# Age bins for stratification
df["age_bin"] = pd.cut(
    df["age"],
    bins=[0, 5, 10, 18, 25, 35, 45, 55, 65, 80, 117],
    labels=False,
    include_lowest=True
)

# Some bins may be small; convert to string for stable stratification
df["age_bin"] = df["age_bin"].astype(str)

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    shuffle=True,
    stratify=df["age_bin"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True,
    stratify=temp_df["age_bin"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))

print("\nTrain age mean/std:", train_df["age"].mean(), train_df["age"].std())
print("Val age mean/std  :", val_df["age"].mean(), val_df["age"].std())
print("Test age mean/std :", test_df["age"].mean(), test_df["age"].std())
"""

Train: 16593
Val  : 3556
Test : 3556

Train age mean/std: 33.31850780449587 19.9110359921922
Val age mean/std  : 33.25956130483689 19.774405774778902
Test age mean/std : 33.26012373453318 19.883885053553964


In [ ]:
# ============================================================
# Save split CSV files for first run
# ============================================================

"""
SAVE_SPLIT_DIR = Path("/content/drive/MyDrive/Datasets/UTKFace_splits")
SAVE_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(SAVE_SPLIT_DIR / "utkface_train.csv", index=False)
val_df.to_csv(SAVE_SPLIT_DIR / "utkface_val.csv", index=False)
test_df.to_csv(SAVE_SPLIT_DIR / "utkface_test.csv", index=False)

print("Saved splits to:", SAVE_SPLIT_DIR)
"""

AGE_MEAN: 33.31850780449587
AGE_STD : 19.9110359921922


In [ ]:
# ============================================================
# Load saved split CSV files, but repair image paths
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

SAVE_SPLIT_DIR = Path("/content/drive/MyDrive/Datasets/UTKFace_splits")

train_df = pd.read_csv(SAVE_SPLIT_DIR / "utkface_train.csv")
val_df   = pd.read_csv(SAVE_SPLIT_DIR / "utkface_val.csv")
test_df  = pd.read_csv(SAVE_SPLIT_DIR / "utkface_test.csv")

print("Loaded split CSVs:")
print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

# ============================================================
# Repair image paths using current kagglehub dataset location
# ============================================================

# This should already exist from your dataset download cell:
# utkface_root = dataset_path / "UTKFace"

utkface_root = Path(utkface_root)

print("Current UTKFace root:")
print(utkface_root)
print("Root exists:", utkface_root.exists())

if not utkface_root.exists():
    raise FileNotFoundError(f"UTKFace root not found: {utkface_root}")

# Build filename -> current full path map
image_files = []
for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    image_files.extend(list(utkface_root.rglob(ext)))

print("Images found under current UTKFace root:", len(image_files))

filename_to_current_path = {}
for p in image_files:
    if p.name not in filename_to_current_path:
        filename_to_current_path[p.name] = str(p)

def repair_df_paths(df, split_name):
    df = df.copy()

    if "file_name" not in df.columns:
        # fallback: extract filename from old image_path
        df["file_name"] = df["image_path"].apply(lambda x: Path(str(x)).name)

    df["image_path"] = df["file_name"].map(filename_to_current_path)

    missing = df["image_path"].isna().sum()

    print(f"{split_name}: missing paths after repair:", missing)

    if missing > 0:
        missing_files = df.loc[df["image_path"].isna(), "file_name"].head(20).tolist()
        print("Example missing files:")
        print(missing_files)
        raise FileNotFoundError(
            f"{missing} files in {split_name} could not be found under {utkface_root}"
        )

    # Final existence check
    exists_mask = df["image_path"].apply(lambda x: Path(x).exists())
    print(f"{split_name}: existing paths:", exists_mask.sum(), "/", len(df))

    if not exists_mask.all():
        bad_examples = df.loc[~exists_mask, "image_path"].head(20).tolist()
        print("Bad path examples:")
        print(bad_examples)
        raise FileNotFoundError(f"Some repaired paths still do not exist in {split_name}")

    return df.reset_index(drop=True)


train_df = repair_df_paths(train_df, "train")
val_df   = repair_df_paths(val_df, "val")
test_df  = repair_df_paths(test_df, "test")

# ============================================================
# Recompute age normalization from repaired train split
# ============================================================

AGE_MEAN = train_df["age"].mean()
AGE_STD  = train_df["age"].std()

print("AGE_MEAN:", AGE_MEAN)
print("AGE_STD :", AGE_STD)

print("\nSample repaired paths:")
print(train_df[["file_name", "image_path", "age"]].head())
print("First path exists:", Path(train_df.iloc[0]["image_path"]).exists())

Loaded split CSVs:
Train: (16593, 6)
Val  : (3556, 6)
Test : (3556, 6)
Current UTKFace root:
/kaggle/input/utkface-new/UTKFace
Root exists: True
Images found under current UTKFace root: 23708
train: missing paths after repair: 0
train: existing paths: 16593 / 16593
val: missing paths after repair: 0
val: existing paths: 3556 / 3556
test: missing paths after repair: 0
test: existing paths: 3556 / 3556
AGE_MEAN: 33.31850780449587
AGE_STD : 19.9110359921922

Sample repaired paths:
                               file_name  \
0  38_0_1_20170113142213931.jpg.chip.jpg   
1  51_1_0_20170104205943165.jpg.chip.jpg   
2   4_0_3_20161220220742914.jpg.chip.jpg   
3  29_1_0_20170104022706451.jpg.chip.jpg   
4  36_0_0_20170116182401789.jpg.chip.jpg   

                                          image_path  age  
0  /kaggle/input/utkface-new/UTKFace/38_0_1_20170...   38  
1  /kaggle/input/utkface-new/UTKFace/51_1_0_20170...   51  
2  /kaggle/input/utkface-new/UTKFace/4_0_3_201612...    4  
3  /kaggle/i

In [ ]:
# ============================================================
# PyTorch Dataset for UTKFace age regression
# ============================================================

from PIL import Image
import torch
from torch.utils.data import Dataset

class UTKFaceAgeDataset(Dataset):
    def __init__(
        self,
        dataframe,
        transform=None,
        return_metadata=False,
        normalize_age=True,
        age_mean=None,
        age_std=None
    ):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.return_metadata = return_metadata
        self.normalize_age = normalize_age
        self.age_mean = age_mean
        self.age_std = age_std

        if self.normalize_age:
            if self.age_mean is None or self.age_std is None:
                raise ValueError("age_mean and age_std are required when normalize_age=True")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = row["image_path"]
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        age_raw = float(row["age"])

        if self.normalize_age:
            age = (age_raw - self.age_mean) / self.age_std
        else:
            age = age_raw

        age = torch.tensor(age, dtype=torch.float32)

        if self.return_metadata:
            metadata = {
                "file_name": row["file_name"],
                "gender": int(row["gender"]),
                "race": int(row["race"]),
                "image_path": image_path,
                "age_raw": age_raw,
            }

            return image, age, metadata

        return image, age

In [ ]:
# ============================================================
# Image transforms
# ============================================================
from torchvision import transforms

target_img_size = 224 # 224

transform_train = transforms.Compose([
    transforms.Resize((target_img_size, target_img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.02
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

transform_val_test = transforms.Compose([
    transforms.Resize((target_img_size, target_img_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# ============================================================
# Create datasets and dataloaders
# ============================================================
from torch.utils.data import DataLoader


batch_size = 32
num_workers = 2

trainset = UTKFaceAgeDataset(
    dataframe=train_df,
    transform=transform_train,
    return_metadata=False,
    normalize_age=True,
    age_mean=AGE_MEAN,
    age_std=AGE_STD
)

valset = UTKFaceAgeDataset(
    dataframe=val_df,
    transform=transform_val_test,
    return_metadata=False,
    normalize_age=True,
    age_mean=AGE_MEAN,
    age_std=AGE_STD
)

testset = UTKFaceAgeDataset(
    dataframe=test_df,
    transform=transform_val_test,
    return_metadata=False,
    normalize_age=True,
    age_mean=AGE_MEAN,
    age_std=AGE_STD
)

trainloader = DataLoader(
    trainset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True
)

valloader = DataLoader(
    valset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

testloader = DataLoader(
    testset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

print("Train batches:", len(trainloader))
print("Val batches  :", len(valloader))
print("Test batches :", len(testloader))

# ============================================================
# Sanity check
# ============================================================

images, ages = next(iter(trainloader))

print("Image batch shape:", images.shape)
print("Age batch shape:", ages.shape)
print("Age min/max:", ages.min().item(), ages.max().item())
print("Age examples:", ages[:10])

Train batches: 519
Val batches  : 112
Test batches : 112
Image batch shape: torch.Size([32, 3, 224, 224])
Age batch shape: torch.Size([32])
Age min/max: -1.6231454610824585 1.340035319328308
Age examples: tensor([-0.2169,  0.2351, -0.5182, -0.3676, -0.3676, -0.3676, -0.4178, -1.3218,
        -0.3676,  1.1391])


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


##### alpha ablation
"""
dropout_ratio = 0.5
model_specs = {}

# Fixed gamma for alpha ablation
signentropy_gamma = 1.0

alpha_values = [
    3,
    5
]

# Add Sign Entropy alpha ablation models
for alpha in alpha_values:
    alpha_name = str(alpha).replace(".", "p")

    model_specs[f"entropy_alpha_{alpha_name}"] = {
        "class": BayesianVGG16_AgeRegression,
        "kwargs": {
            "drop_type": "signentropy",
            "drop_prob": dropout_ratio,
            "alpha": alpha,
            "gamma": signentropy_gamma,
            "pretrained": True,
            "freeze_features": False
        }
    }

print("Models to train:")
for name, spec in model_specs.items():
    print(name, spec["kwargs"])
"""
##### alpha ablation

#### gamma ablation
# ============================================================
# Model specs with Sign Entropy gamma ablation
# ============================================================
"""
dropout_ratio = 0.3
model_specs = {}

signentropy_alpha = 50

gamma_values = [
    0.8,
    0.85,
    0.9,
    0.95,
]


# Add Sign Entropy gamma ablation models
for gamma in gamma_values:
    gamma_name = str(gamma).replace(".", "p")

    model_specs[f"entropy_gamma_{gamma_name}"] = {
        "class": BayesianVGG16_AgeRegression,
        "kwargs": {
            "drop_type": "signentropy",
            "drop_prob": dropout_ratio,
            "alpha": signentropy_alpha,
            "gamma": gamma,
            "pretrained": True,
            "freeze_features": False
        }
    }

print("Models to train:")
for name, spec in model_specs.items():
    print(name, spec["kwargs"])
"""
#### gamma ablation
# ============================================================
# Model specs with Sign Entropy gamma ablation
# ============================================================

####
#### Trials

dropout_ratio = 0.5
model_specs = {
    "entropy_a50_g1": {
        "class": BayesianVGG16_AgeRegression,
        "kwargs": {
            "drop_type": "signentropy",
            "drop_prob": dropout_ratio,
            "alpha": 50.0,
            "gamma": 1.0,
            "pretrained": True,
            "freeze_features": False
        }
    },

    "entropy_a5_g0p1": {
        "class": BayesianVGG16_AgeRegression,
        "kwargs": {
            "drop_type": "signentropy",
            "drop_prob": dropout_ratio,
            "alpha": 5.0,
            "gamma": 0.1,
            "pretrained": True,
            "freeze_features": False
        }
    },

    "entropy_a10_g1": {
        "class": BayesianVGG16_AgeRegression,
        "kwargs": {
            "drop_type": "signentropy",
            "drop_prob": dropout_ratio,
            "alpha": 10.0,
            "gamma": 1.0,
            "pretrained": True,
            "freeze_features": False
        }
    }
}


####
#### Trials




### all models
signentropy_gamma = 1.0
signentropy_alpha = 5.0
dropout_ratio = 0.1
model_specs_all = {
    "plain": {
        "class": BayesianVGG16_AgeRegression,
        "kwargs": {
            "drop_type": None,
            "drop_prob": dropout_ratio,
            "alpha": signentropy_alpha,
            "gamma": signentropy_gamma,
            "pretrained": True,
            "freeze_features": False
        }
    },

    "dropout": {
        "class": BayesianVGG16_AgeRegression,
        "kwargs": {
            "drop_type": "dropout",
            "drop_prob": dropout_ratio,
            "alpha": signentropy_alpha,
            "gamma": signentropy_gamma,
            "pretrained": True,
            "freeze_features": False
        }
    },

    "dropconnect": {
        "class": BayesianVGG16_AgeRegression,
        "kwargs": {
            "drop_type": "dropconnect",
            "drop_prob": dropout_ratio,
            "alpha": signentropy_alpha,
            "gamma": signentropy_gamma,
            "pretrained": True,
            "freeze_features": False
        }
    },

    "entropy": {
        "class": BayesianVGG16_AgeRegression,
        "kwargs": {
            "drop_type": "signentropy",
            "drop_prob": dropout_ratio,
            "alpha": signentropy_alpha,
            "gamma": signentropy_gamma,
            "pretrained": True,
            "freeze_features": False
        }
    },

    "concrete": {
        "class": ConcreteVGG16_AgeRegression,
        "kwargs": {
            "drop_prob": dropout_ratio,
            "pretrained": True,
            "freeze_features": False,
            "weight_regularizer": 1e-6,
            "dropout_regularizer": 1e-5,
            "concrete_temperature": 0.1
        }
    }
}

#all models

print("Models:", list(model_specs.keys()))

Using device: cuda
GPU: NVIDIA A100-SXM4-80GB
Models: ['entropy_a50_g1', 'entropy_a5_g0p1', 'entropy_a10_g1']


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import torch
import torch.nn.functional as F

def denormalize_mean_and_logvar(mean_norm, logvar_norm, age_mean, age_std):
    mean_age = mean_norm * age_std + age_mean
    logvar_age = logvar_norm + 2.0 * np.log(age_std)
    return mean_age, logvar_age


def evaluate_regression_fast(model, data_loader, device, age_mean, age_std):
    model.eval()

    all_targets_norm = []
    all_pred_mean_norm = []
    all_pred_logvar_norm = []

    total_loss = 0.0
    total_batches = 0

    is_bayesian_head = (
        hasattr(model, "regression_head")
        and hasattr(model.regression_head, "weight_mu")
    )

    with torch.no_grad():
        for images, targets_norm in data_loader:
            images = images.to(device)
            targets_norm = targets_norm.float().to(device)

            if is_bayesian_head:
                features = model.forward_features(images)
                head = model.regression_head

                weight = head.weight_mu

                if getattr(model, "drop_type", None) == "dropconnect":
                    weight = weight * (1.0 - model.drop_prob)

                bias = head.bias_mu if hasattr(head, "bias_mu") else None

                out = F.linear(features, weight, bias)

                pred_mean_norm = out[:, 0]
                pred_logvar_norm = out[:, 1]

            else:
                pred_mean_norm, pred_logvar_norm = model(images)

            loss = gaussian_nll_loss(
                pred_mean_norm,
                pred_logvar_norm,
                targets_norm
            )

            total_loss += loss.item()
            total_batches += 1

            all_targets_norm.append(targets_norm.detach().cpu().numpy())
            all_pred_mean_norm.append(pred_mean_norm.detach().cpu().numpy())
            all_pred_logvar_norm.append(pred_logvar_norm.detach().cpu().numpy())

    y_true_norm = np.concatenate(all_targets_norm)
    y_pred_norm = np.concatenate(all_pred_mean_norm)
    pred_logvar_norm = np.concatenate(all_pred_logvar_norm)

    y_true_age = y_true_norm * age_std + age_mean
    y_pred_age, pred_logvar_age = denormalize_mean_and_logvar(
        y_pred_norm,
        pred_logvar_norm,
        age_mean,
        age_std
    )

    mae = mean_absolute_error(y_true_age, y_pred_age)
    rmse = np.sqrt(mean_squared_error(y_true_age, y_pred_age))

    pred_logvar_age = np.clip(pred_logvar_age, -10.0, 10.0)

    nll_age = 0.5 * (
        np.exp(-pred_logvar_age) * (y_true_age - y_pred_age) ** 2
        + pred_logvar_age
    )

    return {
        "loss": total_loss / total_batches,
        "MAE": float(mae),
        "RMSE": float(rmse),
        "NLL": float(np.mean(nll_age)),
    }

In [ ]:
def build_optimizer(model, lr_features=1e-5, lr_head=1e-4, weight_decay=1e-4):
    feature_params = []
    head_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        if name.startswith("features"):
            feature_params.append(param)
        else:
            head_params.append(param)

    optimizer = optim.AdamW(
        [
            {"params": feature_params, "lr": lr_features},
            {"params": head_params, "lr": lr_head},
        ],
        weight_decay=weight_decay
    )

    return optimizer

In [ ]:
def train_one_epoch_regression(
    model,
    trainloader,
    optimizer,
    device,
    epoch,
    num_epochs,
    name="model",
    grad_clip=5.0
):
    model.train()

    total_loss = 0.0
    total_batches = 0

    for images, ages in trainloader:
        images = images.to(device)
        ages = ages.float().to(device)

        optimizer.zero_grad(set_to_none=True)

        pred_mean, pred_log_var = model(
            images,
            current_epoch=epoch,
            total_epochs=num_epochs
        )

        loss = gaussian_nll_loss(
            pred_mean,
            pred_log_var,
            ages
        )

        if name.lower() == "concrete":
            loss = loss + model.cd_regularization

        loss.backward()

        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=grad_clip
            )

        optimizer.step()

        total_loss += loss.item()
        total_batches += 1

    return total_loss / total_batches

In [ ]:
####Training

import copy
import gc
import os
import json
import pandas as pd
import torch.optim as optim
from sklearn.metrics import mean_absolute_error, mean_squared_error


SAVE_DIR = "/content/drive/MyDrive/PhD/Results/dropconnect/20260815/UTKFace/signentropy_age_regression_vgg_utkface_p01_alphaNgammaTest_pretrained1_freeze0"

os.makedirs(SAVE_DIR, exist_ok=True)

CHECKPOINT_PATH = os.path.join(SAVE_DIR, "vgg_age_regression_models.pt")
METADATA_PATH = os.path.join(SAVE_DIR, "vgg_age_regression_metadata.json")

def make_serializable_model_specs(model_specs):
    serializable_specs = {}

    for model_name, spec in model_specs.items():
        serializable_specs[model_name] = {
            "class_name": spec["class"].__name__,
            "kwargs": spec["kwargs"]
        }

    return serializable_specs


max_epochs = 60
patience = 10
min_delta = 1e-4

trained_state_dicts = {}
training_history = {}
best_val_scores = {}

serializable_model_specs = make_serializable_model_specs(model_specs)

for model_name, spec in model_specs.items():

    print("\n" + "=" * 100)
    print(f"Training model: {model_name}")
    print("=" * 100)

    # Create only one model and move only this model to GPU
    model = spec["class"](**spec["kwargs"]).to(device)

    optimizer = build_optimizer(
        model,
        lr_features=1e-5,
        lr_head=1e-4,
        weight_decay=1e-4
    )

    best_val_mae = float("inf")
    best_state = None
    epochs_without_improvement = 0
    history_rows = []

    for epoch in range(1, max_epochs + 1):

        train_loss = train_one_epoch_regression(
            model=model,
            trainloader=trainloader,
            optimizer=optimizer,
            device=device,
            epoch=epoch,
            num_epochs=max_epochs,
            name=model_name,
            grad_clip=5.0
        )


        val_metrics = evaluate_regression_fast(
            model=model,
            data_loader=valloader,
            device=device,
            age_mean=AGE_MEAN,
            age_std=AGE_STD
        )

        current_val_mae = val_metrics["MAE"]

        improved = current_val_mae < (best_val_mae - min_delta)

        if improved:
            best_val_mae = current_val_mae
            epochs_without_improvement = 0

            best_state = copy.deepcopy({
                k: v.detach().cpu()
                for k, v in model.state_dict().items()
            })

            status = "improved"
        else:
            epochs_without_improvement += 1
            status = f"no improvement {epochs_without_improvement}/{patience}"

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_MAE": val_metrics["MAE"],
            "val_RMSE": val_metrics["RMSE"],
            "val_NLL": val_metrics["NLL"],
            "best_val_MAE": best_val_mae,
            "improved": improved,
            "epochs_without_improvement": epochs_without_improvement,
        }

        if model_name.lower() == "concrete":
            row["concrete_p"] = model.get_concrete_dropout_prob()

        history_rows.append(row)

        print(
            f"Epoch [{epoch:03d}/{max_epochs}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val MAE: {val_metrics['MAE']:.4f} | "
            f"Val RMSE: {val_metrics['RMSE']:.4f} | "
            f"Val NLL: {val_metrics['NLL']:.4f} | "
            f"Best MAE: {best_val_mae:.4f} | "
            f"{status}"
        )

        if model_name.lower() == "concrete":
            print(f"Concrete p: {model.get_concrete_dropout_prob():.4f}")

        if epochs_without_improvement >= patience:
            print(
                f"\nEarly stopping {model_name} at epoch {epoch}. "
                f"No validation MAE improvement for {patience} epochs."
            )
            break

    if best_state is None:
        raise RuntimeError(f"No best_state was saved for model: {model_name}")

    trained_state_dicts[model_name] = best_state
    training_history[model_name] = pd.DataFrame(history_rows)
    best_val_scores[model_name] = best_val_mae

    print(f"\nFinished {model_name}")
    print(f"Best validation MAE: {best_val_mae:.4f}")

    # Save checkpoint immediately after each model
    checkpoint = {
        "state_dicts": trained_state_dicts,
        "model_specs": serializable_model_specs,
        "best_val_scores": best_val_scores,
        "dropout_ratio": dropout_ratio,
    }

    torch.save(checkpoint, CHECKPOINT_PATH)

    metadata = {
        "checkpoint_path": CHECKPOINT_PATH,
        "saved_models": list(trained_state_dicts.keys()),
        "model_specs": serializable_model_specs,
        "best_val_scores": best_val_scores,
        "dropout_ratio": dropout_ratio,
    }

    with open(METADATA_PATH, "w") as f:
        json.dump(metadata, f, indent=4)

    print("Checkpoint updated.")
    print("Saved so far:", list(trained_state_dicts.keys()))

    # Remove this model from GPU before next model
    del model
    del optimizer
    torch.cuda.empty_cache()
    gc.collect()


Training model: entropy_a50_g1
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 211MB/s]


Epoch [001/60] | Train Loss: 0.4091 | Val MAE: 8.2679 | Val RMSE: 10.8512 | Val NLL: 2.9221 | Best MAE: 8.2679 | improved
Epoch [002/60] | Train Loss: -0.0285 | Val MAE: 6.8256 | Val RMSE: 9.3534 | Val NLL: 2.6735 | Best MAE: 6.8256 | improved
Epoch [003/60] | Train Loss: -0.2715 | Val MAE: 6.8577 | Val RMSE: 9.1873 | Val NLL: 2.6311 | Best MAE: 6.8256 | no improvement 1/10
Epoch [004/60] | Train Loss: -0.4024 | Val MAE: 6.1498 | Val RMSE: 8.7757 | Val NLL: 2.6437 | Best MAE: 6.1498 | improved
Epoch [005/60] | Train Loss: -0.5160 | Val MAE: 6.0892 | Val RMSE: 8.5465 | Val NLL: 2.6074 | Best MAE: 6.0892 | improved
Epoch [006/60] | Train Loss: -0.5823 | Val MAE: 6.2378 | Val RMSE: 8.4051 | Val NLL: 2.5366 | Best MAE: 6.0892 | no improvement 1/10
Epoch [007/60] | Train Loss: -0.5894 | Val MAE: 6.2158 | Val RMSE: 8.4927 | Val NLL: 2.5545 | Best MAE: 6.0892 | no improvement 2/10
Epoch [008/60] | Train Loss: -0.6181 | Val MAE: 6.0896 | Val RMSE: 8.3976 | Val NLL: 2.4764 | Best MAE: 6.0892 | 

In [ ]:
#SAVE_ROOT = "/content/drive/MyDrive/PhD/Results//dropconnect/20260815/UTKFace/signentropy_age_regression_vgg_utkface_p03_alpha1_gamma1_pretrained1_freeze0"

In [ ]:
trained_state_dicts.keys()

dict_keys(['entropy_a50_g1', 'entropy_a5_g0p1', 'entropy_a10_g1'])

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from scipy.special import logsumexp
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
def denormalize_mc_samples(mean_samples_norm, logvar_samples_norm, age_mean, age_std):
    mean_samples_age = mean_samples_norm * age_std + age_mean
    logvar_samples_age = logvar_samples_norm + 2.0 * np.log(age_std)
    return mean_samples_age, logvar_samples_age


def denormalize_targets(y_true_norm, age_mean, age_std):
    return y_true_norm * age_std + age_mean

In [ ]:
def regression_mc_predict_split_head(
    model,
    data_loader,
    num_samples=50,
    device=None
):
    if device is None:
        device = next(model.parameters()).device

    model.eval()

    all_mean_samples = []
    all_logvar_samples = []
    all_targets = None

    has_split_bayesian_head = (
        hasattr(model, "mean_head")
        and hasattr(model.mean_head, "weight_mu")
        and hasattr(model, "logvar_head")
    )

    with torch.no_grad():

        for sample_idx in range(num_samples):

            mean_list = []
            logvar_list = []
            target_list = []

            if has_split_bayesian_head:
                mean_head = model.mean_head

                weight_mu = mean_head.weight_mu
                weight_sigma = torch.exp(mean_head.weight_log_sigma)

                sampled_weight = weight_mu + torch.randn_like(weight_mu) * weight_sigma

                # Keep this consistent with your classification code:
                # DropConnect uses inference scaling by (1 - p).
                if getattr(model, "drop_type", None) == "dropconnect":
                    sampled_weight = sampled_weight * (1.0 - model.drop_prob)

                if hasattr(mean_head, "bias_mu") and mean_head.bias_mu is not None:
                    bias_mu = mean_head.bias_mu
                    bias_sigma = torch.exp(mean_head.bias_log_sigma)
                    sampled_bias = bias_mu + torch.randn_like(bias_mu) * bias_sigma
                else:
                    sampled_bias = None

            for images, targets in data_loader:
                images = images.to(device)
                targets = targets.float()

                if has_split_bayesian_head:
                    features = model.forward_features(images)

                    pred_mean = F.linear(
                        features,
                        sampled_weight,
                        sampled_bias
                    ).squeeze(1)

                    # Deterministic aleatoric head
                    pred_logvar = model.logvar_head(features).squeeze(1)

                else:
                    # For Concrete model or any non-split model
                    pred_mean, pred_logvar = model(images)

                mean_list.append(pred_mean.detach().cpu().numpy())
                logvar_list.append(pred_logvar.detach().cpu().numpy())

                if sample_idx == 0:
                    target_list.append(targets.detach().cpu().numpy())

            all_mean_samples.append(np.concatenate(mean_list, axis=0))
            all_logvar_samples.append(np.concatenate(logvar_list, axis=0))

            if sample_idx == 0:
                all_targets = np.concatenate(target_list, axis=0)

    mean_samples = np.stack(all_mean_samples, axis=0)
    logvar_samples = np.stack(all_logvar_samples, axis=0)

    return mean_samples, logvar_samples, all_targets

In [ ]:
def compute_mc_regression_metrics(
    mean_samples_age,
    logvar_samples_age,
    y_true_age,
    eps=1e-8
):
    mean_samples_age = np.asarray(mean_samples_age)
    logvar_samples_age = np.asarray(logvar_samples_age)
    y_true_age = np.asarray(y_true_age).reshape(-1)

    logvar_samples_age = np.clip(logvar_samples_age, -10.0, 10.0)

    pred_mean_age = np.mean(mean_samples_age, axis=0)

    mae = mean_absolute_error(y_true_age, pred_mean_age)
    rmse = np.sqrt(mean_squared_error(y_true_age, pred_mean_age))

    aleatoric_per_sample = np.mean(
        np.exp(logvar_samples_age),
        axis=0
    )

    epistemic_per_sample = np.var(
        mean_samples_age,
        axis=0
    )

    total_uncertainty_per_sample = aleatoric_per_sample + epistemic_per_sample

    T = mean_samples_age.shape[0]

    log_probs = []

    for t in range(T):
        mu_t = mean_samples_age[t]
        logvar_t = logvar_samples_age[t]
        var_t = np.exp(logvar_t) + eps

        log_prob_t = -0.5 * (
            np.log(2.0 * np.pi)
            + np.log(var_t)
            + ((y_true_age - mu_t) ** 2) / var_t
        )

        log_probs.append(log_prob_t)

    log_probs = np.stack(log_probs, axis=0)

    predictive_log_prob = logsumexp(log_probs, axis=0) - np.log(T)
    nll_per_sample = -predictive_log_prob

    return {
        "y_true_age": y_true_age,
        "pred_mean_age": pred_mean_age,

        "MAE": float(mae),
        "RMSE": float(rmse),
        "NLL": float(np.mean(nll_per_sample)),

        "aleatoric_per_sample": aleatoric_per_sample,
        "epistemic_per_sample": epistemic_per_sample,
        "total_uncertainty_per_sample": total_uncertainty_per_sample,

        "mean_aleatoric": float(np.mean(aleatoric_per_sample)),
        "mean_epistemic": float(np.mean(epistemic_per_sample)),
        "mean_total_uncertainty": float(np.mean(total_uncertainty_per_sample)),

        "nll_per_sample": nll_per_sample,
    }

In [ ]:
def prediction_interval_coverage_from_mc(metrics_result):
    y_true = metrics_result["y_true_age"]
    pred_mean = metrics_result["pred_mean_age"]

    total_var = metrics_result["total_uncertainty_per_sample"]
    total_std = np.sqrt(total_var + 1e-8)

    intervals = {
        "68": 1.0,
        "90": 1.645,
        "95": 1.96,
    }

    results = {}

    for level, z in intervals.items():
        lower = pred_mean - z * total_std
        upper = pred_mean + z * total_std

        covered = (y_true >= lower) & (y_true <= upper)

        picp = np.mean(covered)
        mpiw = np.mean(upper - lower)

        nominal = float(level) / 100.0

        results[level] = {
            "PICP": float(picp),
            "MPIW": float(mpiw),
            "coverage_error": float(abs(picp - nominal)),
        }

    return results

In [ ]:
def evaluate_one_model_mc_test_split_head(
    model,
    testloader,
    device,
    age_mean,
    age_std,
    num_mc_samples=50
):
    mean_samples_norm, logvar_samples_norm, y_true_norm = regression_mc_predict_split_head(
        model=model,
        data_loader=testloader,
        num_samples=num_mc_samples,
        device=device
    )

    mean_samples_age, logvar_samples_age = denormalize_mc_samples(
        mean_samples_norm,
        logvar_samples_norm,
        age_mean,
        age_std
    )

    y_true_age = denormalize_targets(
        y_true_norm,
        age_mean,
        age_std
    )

    metrics_result = compute_mc_regression_metrics(
        mean_samples_age=mean_samples_age,
        logvar_samples_age=logvar_samples_age,
        y_true_age=y_true_age
    )

    coverage_result = prediction_interval_coverage_from_mc(
        metrics_result
    )

    result = {
        **metrics_result,
        "coverage": coverage_result,
        "mean_samples_age": mean_samples_age,
        "logvar_samples_age": logvar_samples_age,
    }

    return result

In [ ]:
### MC without temperature scaling
num_mc_samples = 50

mc_test_results = {}
summary_rows = []

for model_name, state_dict in trained_state_dicts.items():

    print("\n" + "=" * 100)
    print("MC Testing:", model_name)
    print("=" * 100)

    spec = model_specs[model_name]

    model = spec["class"](**spec["kwargs"])
    model.load_state_dict(state_dict, strict=True)
    model.to(device)
    model.eval()

    result = evaluate_one_model_mc_test_split_head(
        model=model,
        testloader=testloader,
        device=device,
        age_mean=AGE_MEAN,
        age_std=AGE_STD,
        num_mc_samples=num_mc_samples
    )

    mc_test_results[model_name] = result

    row = {
        "Method": model_name,

        "MAE": result["MAE"],
        "RMSE": result["RMSE"],
        "NLL": result["NLL"],

        "Aleatoric_U": result["mean_aleatoric"],
        "Epistemic_U": result["mean_epistemic"],
        "Total_U": result["mean_total_uncertainty"],

        "PICP_68": result["coverage"]["68"]["PICP"],
        "PICP_90": result["coverage"]["90"]["PICP"],
        "PICP_95": result["coverage"]["95"]["PICP"],

        "Coverage_Error_68": result["coverage"]["68"]["coverage_error"],
        "Coverage_Error_90": result["coverage"]["90"]["coverage_error"],
        "Coverage_Error_95": result["coverage"]["95"]["coverage_error"],

        "MPIW_68": result["coverage"]["68"]["MPIW"],
        "MPIW_90": result["coverage"]["90"]["MPIW"],
        "MPIW_95": result["coverage"]["95"]["MPIW"],
    }

    if model_name.lower() == "concrete" and hasattr(model, "get_concrete_dropout_prob"):
        row["Concrete_p"] = model.get_concrete_dropout_prob()
    else:
        row["Concrete_p"] = np.nan

    summary_rows.append(row)

    print("MAE:", result["MAE"])
    print("RMSE:", result["RMSE"])
    print("NLL:", result["NLL"])

    print("Aleatoric U:", result["mean_aleatoric"])
    print("Epistemic U:", result["mean_epistemic"])
    print("Total U:", result["mean_total_uncertainty"])

    print("PICP 68:", result["coverage"]["68"])
    print("PICP 90:", result["coverage"]["90"])
    print("PICP 95:", result["coverage"]["95"])

    if model_name.lower() == "concrete" and hasattr(model, "get_concrete_dropout_prob"):
        print("Concrete p:", model.get_concrete_dropout_prob())

    del model
    torch.cuda.empty_cache()
    gc.collect()

mc_test_summary_df = pd.DataFrame(summary_rows)
mc_test_summary_df

In [ ]:
########################### MC Code for Scalar validation-fitted log-variance temperature/shift.

import numpy as np
from scipy.special import logsumexp
from scipy.optimize import minimize_scalar
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.special import ndtr


def mc_gaussian_nll_with_logvar_shift(
    mean_samples_age,
    logvar_samples_age,
    y_true_age,
    logvar_shift=0.0,
    eps=1e-8
):
    """
    mean_samples_age:   (T, N)
    logvar_samples_age: (T, N)
    y_true_age:         (N,)

    Applies:
        calibrated_logvar = logvar + logvar_shift
    """

    mean_samples_age = np.asarray(mean_samples_age)
    logvar_samples_age = np.asarray(logvar_samples_age)
    y_true_age = np.asarray(y_true_age).reshape(-1)

    calibrated_logvar = logvar_samples_age + logvar_shift
    calibrated_logvar = np.clip(calibrated_logvar, -10.0, 10.0)

    T = mean_samples_age.shape[0]

    log_probs = []

    for t in range(T):
        mu_t = mean_samples_age[t]
        logvar_t = calibrated_logvar[t]
        var_t = np.exp(logvar_t) + eps

        log_prob_t = -0.5 * (
            np.log(2.0 * np.pi)
            + np.log(var_t)
            + ((y_true_age - mu_t) ** 2) / var_t
        )

        log_probs.append(log_prob_t)

    log_probs = np.stack(log_probs, axis=0)

    predictive_log_prob = logsumexp(log_probs, axis=0) - np.log(T)
    nll = -np.mean(predictive_log_prob)

    return float(nll)


def fit_logvar_shift_on_validation(
    mean_samples_val_age,
    logvar_samples_val_age,
    y_val_age,
    shift_bounds=(-5.0, 5.0)
):
    """
    Finds scalar c that minimizes validation NLL:

        logvar_calibrated = logvar + c
    """

    def objective(c):
        return mc_gaussian_nll_with_logvar_shift(
            mean_samples_age=mean_samples_val_age,
            logvar_samples_age=logvar_samples_val_age,
            y_true_age=y_val_age,
            logvar_shift=c
        )

    result = minimize_scalar(
        objective,
        bounds=shift_bounds,
        method="bounded"
    )

    best_shift = float(result.x)
    best_val_nll = float(result.fun)

    return best_shift, best_val_nll



def age_bin_brier_score_from_mc_gaussian(
    mean_samples_age,
    logvar_samples_age,
    y_true_age,
    age_bin_edges=None,
    eps=1e-8
):
    """
    Computes multiclass age-bin Brier score from MC Gaussian predictive distribution.

    mean_samples_age:   (T, N)
    logvar_samples_age: (T, N)
    y_true_age:         (N,)
    """

    if age_bin_edges is None:
        age_bin_edges = np.array([0, 5, 10, 18, 25, 35, 45, 55, 65, 80, 117], dtype=float)
    else:
        age_bin_edges = np.asarray(age_bin_edges, dtype=float)

    mean_samples_age = np.asarray(mean_samples_age)
    logvar_samples_age = np.asarray(logvar_samples_age)
    y_true_age = np.asarray(y_true_age).reshape(-1)

    T, N = mean_samples_age.shape
    K = len(age_bin_edges) - 1

    sigma_samples_age = np.sqrt(np.exp(logvar_samples_age) + eps)

    bin_probs = np.zeros((N, K), dtype=np.float64)

    for k in range(K):
        lower = age_bin_edges[k]
        upper = age_bin_edges[k + 1]

        z_upper = (upper - mean_samples_age) / sigma_samples_age
        z_lower = (lower - mean_samples_age) / sigma_samples_age

        prob_t = ndtr(z_upper) - ndtr(z_lower)
        prob_t = np.clip(prob_t, 0.0, 1.0)

        bin_probs[:, k] = np.mean(prob_t, axis=0)

    # Normalize small numerical errors
    bin_probs = bin_probs / (bin_probs.sum(axis=1, keepdims=True) + eps)

    true_bins = np.digitize(y_true_age, age_bin_edges[1:-1], right=False)
    true_bins = np.clip(true_bins, 0, K - 1)

    y_onehot = np.zeros((N, K), dtype=np.float64)
    y_onehot[np.arange(N), true_bins] = 1.0

    brier = np.mean(np.sum((bin_probs - y_onehot) ** 2, axis=1))

    return float(brier)


def compute_mc_regression_metrics_calibrated(
    mean_samples_age,
    logvar_samples_age,
    y_true_age,
    logvar_shift=0.0,
    eps=1e-8
):
    mean_samples_age = np.asarray(mean_samples_age)
    logvar_samples_age = np.asarray(logvar_samples_age)
    y_true_age = np.asarray(y_true_age).reshape(-1)

    calibrated_logvar = logvar_samples_age + logvar_shift
    calibrated_logvar = np.clip(calibrated_logvar, -10.0, 10.0)

    pred_mean_age = np.mean(mean_samples_age, axis=0)

    mae = mean_absolute_error(y_true_age, pred_mean_age)
    rmse = np.sqrt(mean_squared_error(y_true_age, pred_mean_age))

    aleatoric_per_sample = np.mean(
        np.exp(calibrated_logvar),
        axis=0
    )

    epistemic_per_sample = np.var(
        mean_samples_age,
        axis=0
    )

    total_uncertainty_per_sample = aleatoric_per_sample + epistemic_per_sample

    nll = mc_gaussian_nll_with_logvar_shift(
        mean_samples_age=mean_samples_age,
        logvar_samples_age=logvar_samples_age,
        y_true_age=y_true_age,
        logvar_shift=logvar_shift,
        eps=eps
    )

    agebin_brier = age_bin_brier_score_from_mc_gaussian(
        mean_samples_age=mean_samples_age,
        logvar_samples_age=calibrated_logvar,
        y_true_age=y_true_age
    )

    return {
        "y_true_age": y_true_age,
        "pred_mean_age": pred_mean_age,

        "MAE": float(mae),
        "RMSE": float(rmse),
        "NLL": float(nll),
        "AgeBin_Brier": float(agebin_brier),

        "aleatoric_per_sample": aleatoric_per_sample,
        "epistemic_per_sample": epistemic_per_sample,
        "total_uncertainty_per_sample": total_uncertainty_per_sample,

        "mean_aleatoric": float(np.mean(aleatoric_per_sample)),
        "mean_epistemic": float(np.mean(epistemic_per_sample)),
        "mean_total_uncertainty": float(np.mean(total_uncertainty_per_sample)),

        "logvar_shift": float(logvar_shift),
        "variance_multiplier": float(np.exp(logvar_shift)),
        "std_multiplier": float(np.exp(0.5 * logvar_shift)),
    }


def prediction_interval_coverage_from_calibrated_metrics(metrics_result):
    y_true = metrics_result["y_true_age"]
    pred_mean = metrics_result["pred_mean_age"]

    total_var = metrics_result["total_uncertainty_per_sample"]
    total_std = np.sqrt(total_var + 1e-8)

    intervals = {
        "68": 1.0,
        "90": 1.645,
        "95": 1.96,
    }

    results = {}

    for level, z in intervals.items():
        lower = pred_mean - z * total_std
        upper = pred_mean + z * total_std

        covered = (y_true >= lower) & (y_true <= upper)

        picp = np.mean(covered)
        mpiw = np.mean(upper - lower)

        nominal = float(level) / 100.0

        results[level] = {
            "PICP": float(picp),
            "MPIW": float(mpiw),
            "coverage_error": float(abs(picp - nominal)),
        }

    return results



def evaluate_model_with_validation_logvar_calibration(
    model,
    valloader,
    testloader,
    device,
    age_mean,
    age_std,
    num_mc_samples=50,
    shift_bounds=(-5.0, 5.0)
):
    # --------------------------------------------------------
    # 1. MC prediction on validation set
    # --------------------------------------------------------
    mean_val_norm, logvar_val_norm, y_val_norm = regression_mc_predict_split_head(
        model=model,
        data_loader=valloader,
        num_samples=num_mc_samples,
        device=device
    )

    mean_val_age, logvar_val_age = denormalize_mc_samples(
        mean_val_norm,
        logvar_val_norm,
        age_mean,
        age_std
    )

    y_val_age = denormalize_targets(
        y_val_norm,
        age_mean,
        age_std
    )

    # --------------------------------------------------------
    # 2. Fit scalar log-variance shift on validation set
    # --------------------------------------------------------
    best_shift, best_val_nll = fit_logvar_shift_on_validation(
        mean_samples_val_age=mean_val_age,
        logvar_samples_val_age=logvar_val_age,
        y_val_age=y_val_age,
        shift_bounds=shift_bounds
    )

    # --------------------------------------------------------
    # 3. MC prediction on test set
    # --------------------------------------------------------
    mean_test_norm, logvar_test_norm, y_test_norm = regression_mc_predict_split_head(
        model=model,
        data_loader=testloader,
        num_samples=num_mc_samples,
        device=device
    )

    mean_test_age, logvar_test_age = denormalize_mc_samples(
        mean_test_norm,
        logvar_test_norm,
        age_mean,
        age_std
    )

    y_test_age = denormalize_targets(
        y_test_norm,
        age_mean,
        age_std
    )

    # --------------------------------------------------------
    # 4. Test metrics using calibrated log variance
    # --------------------------------------------------------
    test_metrics = compute_mc_regression_metrics_calibrated(
        mean_samples_age=mean_test_age,
        logvar_samples_age=logvar_test_age,
        y_true_age=y_test_age,
        logvar_shift=best_shift
    )

    test_coverage = prediction_interval_coverage_from_calibrated_metrics(
        test_metrics
    )

    return {
        "best_logvar_shift": best_shift,
        "best_val_nll": best_val_nll,
        "test_metrics": test_metrics,
        "test_coverage": test_coverage,
    }


import gc
import pandas as pd
import torch

num_mc_samples = 50

calibrated_results = {}
calibrated_summary_rows = []

for model_name, state_dict in trained_state_dicts.items():

    print("\n" + "=" * 100)
    print("Validation-fitted logvar calibration:", model_name)
    print("=" * 100)

    spec = model_specs[model_name]

    model = spec["class"](**spec["kwargs"])
    model.load_state_dict(state_dict, strict=True)
    model.to(device)
    model.eval()

    result = evaluate_model_with_validation_logvar_calibration(
        model=model,
        valloader=valloader,
        testloader=testloader,
        device=device,
        age_mean=AGE_MEAN,
        age_std=AGE_STD,
        num_mc_samples=num_mc_samples,
        shift_bounds=(-5.0, 5.0)
    )

    calibrated_results[model_name] = result

    metrics = result["test_metrics"]
    coverage = result["test_coverage"]

    row = {
        "Method": model_name,

        "LogVar_Shift": result["best_logvar_shift"],
        "Variance_Multiplier": metrics["variance_multiplier"],
        "Std_Multiplier": metrics["std_multiplier"],
        "Val_NLL_after_shift": result["best_val_nll"],

        "Test_MAE": metrics["MAE"],
        "Test_RMSE": metrics["RMSE"],
        "Test_NLL": metrics["NLL"],
        "AgeBin_Brier": metrics["AgeBin_Brier"],

        "Aleatoric_U": metrics["mean_aleatoric"],
        "Epistemic_U": metrics["mean_epistemic"],
        "Total_U": metrics["mean_total_uncertainty"],

        "PICP_68": coverage["68"]["PICP"],
        "PICP_90": coverage["90"]["PICP"],
        "PICP_95": coverage["95"]["PICP"],

        "Coverage_Error_68": coverage["68"]["coverage_error"],
        "Coverage_Error_90": coverage["90"]["coverage_error"],
        "Coverage_Error_95": coverage["95"]["coverage_error"],

        "MPIW_68": coverage["68"]["MPIW"],
        "MPIW_90": coverage["90"]["MPIW"],
        "MPIW_95": coverage["95"]["MPIW"],
    }

    if model_name.lower() == "concrete" and hasattr(model, "get_concrete_dropout_prob"):
        row["Concrete_p"] = model.get_concrete_dropout_prob()
    else:
        row["Concrete_p"] = np.nan

    calibrated_summary_rows.append(row)

    print("Best logvar shift:", result["best_logvar_shift"])
    print("Variance multiplier:", metrics["variance_multiplier"])
    print("Std multiplier:", metrics["std_multiplier"])

    print("Test MAE:", metrics["MAE"])
    print("Test RMSE:", metrics["RMSE"])
    print("Test NLL:", metrics["NLL"])
    print("Age-bin Brier:", metrics["AgeBin_Brier"])

    print("PICP 68:", coverage["68"])
    print("PICP 90:", coverage["90"])
    print("PICP 95:", coverage["95"])

    del model
    torch.cuda.empty_cache()
    gc.collect()

calibrated_summary_df = pd.DataFrame(calibrated_summary_rows)
calibrated_summary_df


Validation-fitted logvar calibration: entropy_a50_g1
Best logvar shift: 1.2900281876662012
Variance multiplier: 3.6328889569708416
Std multiplier: 1.906013892124305
Test MAE: 5.222356104420971
Test RMSE: 7.427846629597965
Test NLL: 3.379336741094379
Age-bin Brier: 0.5939137432604036
PICP 68: {'PICP': 0.7606861642294713, 'MPIW': 15.006343100273863, 'coverage_error': 0.08068616422947128}
PICP 90: {'PICP': 0.9147919010123734, 'MPIW': 24.685434399950502, 'coverage_error': 0.014791901012373376}
PICP 95: {'PICP': 0.9496625421822272, 'MPIW': 29.412432476536768, 'coverage_error': 0.0003374578177727239}

Validation-fitted logvar calibration: entropy_a5_g0p1
Best logvar shift: 2.9766561107228386
Variance multiplier: 19.622092713808765
Std multiplier: 4.429683139210836
Test MAE: 4.962776080103606
Test RMSE: 7.1770901972676935
Test NLL: 3.2497501665492363
Age-bin Brier: 0.550017067677218
PICP 68: {'PICP': 0.7767154105736783, 'MPIW': 14.003363885522402, 'coverage_error': 0.09671541057367827}
PICP 

,Method,LogVar_Shift,Variance_Multiplier,Std_Multiplier,Val_NLL_after_shift,Test_MAE,Test_RMSE,Test_NLL,AgeBin_Brier,Aleatoric_U,...,PICP_68,PICP_90,PICP_95,Coverage_Error_68,Coverage_Error_90,Coverage_Error_95,MPIW_68,MPIW_90,MPIW_95,Concrete_p
0,entropy_a50_g1,1.290028,3.632889,1.906014,3.385349,5.222356,7.427847,3.379337,0.593914,62.197022,...,0.760686,0.914792,0.949663,0.080686,0.014792,0.000337,15.006343,24.685434,29.412432,NaN
1,entropy_a5_g0p1,2.976656,19.622093,4.429683,3.280599,4.962776,7.177090,3.249750,0.550017,57.444199,...,0.776715,0.920135,0.953881,0.096715,0.020135,0.003881,14.003364,23.035534,27.446593,NaN
2,entropy_a10_g1,2.943993,18.991527,4.357927,3.314573,4.916305,7.108827,3.263093,0.555559,66.291413,...,0.782621,0.929978,0.954724,0.102621,0.029978,0.004724,14.775616,24.305888,28.960206,NaN


In [ ]:
cols = [
    "Method",
    "Test_MAE",
    "Test_NLL",
    "AgeBin_Brier",
    "Aleatoric_U",
    "Epistemic_U",
    "Total_U",
    "Coverage_Error_68",
    "Coverage_Error_90",
    "Coverage_Error_95",
    "MPIW_68",
    "MPIW_90",
    "MPIW_95"
]

calibrated_summary_df[cols]

,Method,Test_MAE,Test_NLL,AgeBin_Brier,Aleatoric_U,Epistemic_U,Total_U,Coverage_Error_68,Coverage_Error_90,Coverage_Error_95,MPIW_68,MPIW_90,MPIW_95
0,entropy_a50_g1,5.222356,3.379337,0.593914,62.197022,0.690281,62.887303,0.080686,0.014792,0.000337,15.006343,24.685434,29.412432
1,entropy_a5_g0p1,4.962776,3.249750,0.550017,57.444199,0.172151,57.616350,0.096715,0.020135,0.003881,14.003364,23.035534,27.446593
2,entropy_a10_g1,4.916305,3.263093,0.555559,66.291413,0.169141,66.460554,0.102621,0.029978,0.004724,14.775616,24.305888,28.960206


In [ ]:
#### save the MC results
import os

os.makedirs(SAVE_DIR, exist_ok=True)

calibrated_summary_path = os.path.join(
    SAVE_DIR,
    "calibrated_summary_df.csv"
)

calibrated_summary_df.to_csv(calibrated_summary_path, index=False)

print("Saved calibrated summary to:")
print(calibrated_summary_path)

Saved calibrated summary to:
/content/drive/MyDrive/PhD/Results/dropconnect/20260815/UTKFace/signentropy_age_regression_vgg_utkface_p01_alphaNgammaTest_pretrained1_freeze0/calibrated_summary_df.csv


In [ ]:
### reload MC resuls
import pandas as pd
import os

calibrated_summary_df_new = pd.read_csv(
    os.path.join(SAVE_DIR, "calibrated_summary_df.csv")
)

In [ ]:
import os
import json
import torch
import pandas as pd

# ============================================================
# Save trained models separately
# ============================================================

#SAVE_ROOT = "/content/drive/MyDrive/PhD/Results//dropconnect/20260815/UTKFace/signentropy_age_regression_vgg_utkface_p03_alpha1_gamma1_pretrained0_freeze0"

MODEL_DIR = os.path.join(SAVE_DIR, "saved_models")
HISTORY_DIR = os.path.join(SAVE_DIR, "training_history")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(HISTORY_DIR, exist_ok=True)

MASTER_CHECKPOINT_PATH = os.path.join(SAVE_DIR, "all_models_checkpoint.pt")
METADATA_PATH = os.path.join(SAVE_DIR, "metadata.json")


def make_serializable_model_specs(model_specs):
    serializable_specs = {}

    for model_name, spec in model_specs.items():
        serializable_specs[model_name] = {
            "class_name": spec["class"].__name__,
            "kwargs": spec["kwargs"]
        }

    return serializable_specs


serializable_model_specs = make_serializable_model_specs(model_specs)

# ============================================================
# Save each model separately
# ============================================================

saved_model_paths = {}

for model_name, state_dict in trained_state_dicts.items():

    model_path = os.path.join(MODEL_DIR, f"{model_name}_state_dict.pt")

    single_model_checkpoint = {
        "model_name": model_name,
        "state_dict": state_dict,
        "model_spec": serializable_model_specs[model_name],
        "best_val_score": best_val_scores.get(model_name, None),
        "dropout_ratio": dropout_ratio,
        "age_mean": float(AGE_MEAN),
        "age_std": float(AGE_STD),
    }

    torch.save(single_model_checkpoint, model_path)

    saved_model_paths[model_name] = model_path

    print(f"Saved {model_name} to:")
    print(model_path)

    # Save training history if available
    if model_name in training_history:
        history_path = os.path.join(HISTORY_DIR, f"{model_name}_history.csv")
        training_history[model_name].to_csv(history_path, index=False)
        print(f"Saved history for {model_name} to:")
        print(history_path)

# ============================================================
# Save master checkpoint with all models
# ============================================================

master_checkpoint = {
    "state_dicts": trained_state_dicts,
    "model_specs": serializable_model_specs,
    "best_val_scores": best_val_scores,
    "saved_model_paths": saved_model_paths,
    "dropout_ratio": dropout_ratio,
    "age_mean": float(AGE_MEAN),
    "age_std": float(AGE_STD),
}

torch.save(master_checkpoint, MASTER_CHECKPOINT_PATH)

metadata = {
    "save_dir": SAVE_DIR,
    "model_dir": MODEL_DIR,
    "history_dir": HISTORY_DIR,
    "master_checkpoint_path": MASTER_CHECKPOINT_PATH,
    "saved_model_paths": saved_model_paths,
    "model_specs": serializable_model_specs,
    "best_val_scores": best_val_scores,
    "dropout_ratio": dropout_ratio,
    "age_mean": float(AGE_MEAN),
    "age_std": float(AGE_STD),
}

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=4)

print("\nAll models saved.")
print("Master checkpoint:", MASTER_CHECKPOINT_PATH)
print("Metadata:", METADATA_PATH)

Saved entropy_a50_g1 to:
/content/drive/MyDrive/PhD/Results/dropconnect/20260815/UTKFace/signentropy_age_regression_vgg_utkface_p01_alphaNgammaTest_pretrained1_freeze0/saved_models/entropy_a50_g1_state_dict.pt
Saved history for entropy_a50_g1 to:
/content/drive/MyDrive/PhD/Results/dropconnect/20260815/UTKFace/signentropy_age_regression_vgg_utkface_p01_alphaNgammaTest_pretrained1_freeze0/training_history/entropy_a50_g1_history.csv
Saved entropy_a5_g0p1 to:
/content/drive/MyDrive/PhD/Results/dropconnect/20260815/UTKFace/signentropy_age_regression_vgg_utkface_p01_alphaNgammaTest_pretrained1_freeze0/saved_models/entropy_a5_g0p1_state_dict.pt
Saved history for entropy_a5_g0p1 to:
/content/drive/MyDrive/PhD/Results/dropconnect/20260815/UTKFace/signentropy_age_regression_vgg_utkface_p01_alphaNgammaTest_pretrained1_freeze0/training_history/entropy_a5_g0p1_history.csv
Saved entropy_a10_g1 to:
/content/drive/MyDrive/PhD/Results/dropconnect/20260815/UTKFace/signentropy_age_regression_vgg_utkface_

In [ ]:
# ============================================================
# Load and Reconstruct variables needed for MC evaluation
# ============================================================

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

SAVE_DIR = "/content/drive/MyDrive/PhD/Results/dropconnect/20260815/UTKFace/signentropy_age_regression_vgg_utkface_p01_alpha5_gamma1_pretrained1_freeze0"

MODEL_DIR = os.path.join(SAVE_DIR, "saved_models")
HISTORY_DIR = os.path.join(SAVE_DIR, "training_history")

MASTER_CHECKPOINT_PATH = os.path.join(SAVE_DIR, "all_models_checkpoint.pt")
METADATA_PATH = os.path.join(SAVE_DIR, "metadata.json")

ckpt = torch.load(MASTER_CHECKPOINT_PATH, map_location="cpu", weights_only=False)

print("Saved models:", list(ckpt["state_dicts"].keys()))
print("Best val scores:", ckpt["best_val_scores"])
print("AGE_MEAN:", ckpt["age_mean"])
print("AGE_STD :", ckpt["age_std"])

# ------------------------------------------------------------
# These are directly restored
# ------------------------------------------------------------

trained_state_dicts = ckpt["state_dicts"]
best_val_scores = ckpt["best_val_scores"]
dropout_ratio = ckpt["dropout_ratio"]

AGE_MEAN = ckpt["age_mean"]
AGE_STD = ckpt["age_std"]

# ------------------------------------------------------------
# Rebuild model_specs from saved serializable specs
# ------------------------------------------------------------
# This assumes your model class definitions have already been run
# in the notebook before this cell.
# For example:
# class VGG16AgeRegressor(...)
# class ConcreteVGG16AgeRegressor(...)

saved_model_specs = ckpt["model_specs"]

model_specs = {}

for model_name, saved_spec in saved_model_specs.items():

    class_name = saved_spec["class_name"]
    kwargs = saved_spec["kwargs"]

    if class_name not in globals():
        raise NameError(
            f"Class {class_name} is not defined in the notebook. "
            f"Run the cell where {class_name} is defined before this cell."
        )

    model_class = globals()[class_name]

    model_specs[model_name] = {
        "class": model_class,
        "kwargs": kwargs
    }

print("\nReconstructed model_specs:")
for name, spec in model_specs.items():
    print(name, "->", spec["class"].__name__, spec["kwargs"])

Using device: cuda
Saved models: ['plain', 'dropout', 'dropconnect', 'entropy', 'concrete']
Best val scores: {'plain': 5.047354296286652, 'dropout': 5.038653792982544, 'dropconnect': 4.90892373587636, 'entropy': 5.037154452539564, 'concrete': 4.946462758014246}
AGE_MEAN: 33.31850780449587
AGE_STD : 19.9110359921922

Reconstructed model_specs:
plain -> BayesianVGG16_AgeRegression {'drop_type': None, 'drop_prob': 0.1, 'alpha': 5.0, 'gamma': 1.0, 'pretrained': True, 'freeze_features': False}
dropout -> BayesianVGG16_AgeRegression {'drop_type': 'dropout', 'drop_prob': 0.1, 'alpha': 5.0, 'gamma': 1.0, 'pretrained': True, 'freeze_features': False}
dropconnect -> BayesianVGG16_AgeRegression {'drop_type': 'dropconnect', 'drop_prob': 0.1, 'alpha': 5.0, 'gamma': 1.0, 'pretrained': True, 'freeze_features': False}
entropy -> BayesianVGG16_AgeRegression {'drop_type': 'signentropy', 'drop_prob': 0.1, 'alpha': 5.0, 'gamma': 1.0, 'pretrained': True, 'freeze_features': False}
concrete -> ConcreteVGG16_

In [ ]:
SAVE_DIR

'/content/drive/MyDrive/PhD/Results/dropconnect/20260815/UTKFace/signentropy_age_regression_vgg_utkface_p01_alpha5_gamma1_pretrained1_freeze0'

In [ ]:
HISTORY_DIR = os.path.join(SAVE_DIR, "training_history")
os.makedirs(HISTORY_DIR, exist_ok=True)

for model_name, hist_df in training_history.items():
    history_path = os.path.join(HISTORY_DIR, f"{model_name}_training_history.csv")
    hist_df.to_csv(history_path, index=False)
    print("Saved:", history_path)

Saved: /content/drive/MyDrive/PhD/Results/dropconnect/20260815/UTKFace/signentropy_age_regression_vgg_utkface_p01_alpha5_gamma1_pretrained1_freeze0/training_history/plain_training_history.csv
Saved: /content/drive/MyDrive/PhD/Results/dropconnect/20260815/UTKFace/signentropy_age_regression_vgg_utkface_p01_alpha5_gamma1_pretrained1_freeze0/training_history/dropout_training_history.csv
Saved: /content/drive/MyDrive/PhD/Results/dropconnect/20260815/UTKFace/signentropy_age_regression_vgg_utkface_p01_alpha5_gamma1_pretrained1_freeze0/training_history/dropconnect_training_history.csv
Saved: /content/drive/MyDrive/PhD/Results/dropconnect/20260815/UTKFace/signentropy_age_regression_vgg_utkface_p01_alpha5_gamma1_pretrained1_freeze0/training_history/entropy_training_history.csv
Saved: /content/drive/MyDrive/PhD/Results/dropconnect/20260815/UTKFace/signentropy_age_regression_vgg_utkface_p01_alpha5_gamma1_pretrained1_freeze0/training_history/concrete_training_history.csv
